## filter out pages with 21+ lines detected

In [14]:
import os
import cv2
import numpy as np

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'seam_carved_line_images6_case_and_ayer_imgs\segmentation_summary.csv')
df.head()

,image,lines_detected,status
0,991411418805867_Ayer_MS_370_00040-1.jpg,25,OK
1,991411418805867_Ayer_MS_370_00041-1.jpg,23,OK
2,991411418805867_Ayer_MS_370_00042-1.jpg,25,OK
3,991411418805867_Ayer_MS_370_00043-1.jpg,25,OK
4,991411418805867_Ayer_MS_370_00044-1.jpg,25,OK


In [ ]:
for i in range(len(df)):
    if df.loc[i, 'lines_detected'] > 21:
        print(f"Page {df.loc[i, 'image']} has {df.loc[i, 'lines_detected']} lines detected. Consider including it in training.")
        df1 = df[df['lines_detected'] > 21]

## filtering functions

In [13]:

def estimate_background_color(img_bgr, bright_thresh=200):
    """
    Estimate background color from bright pixels.
    img_bgr is a colour image read by OpenCV.
    """
    original = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = gray >= bright_thresh

    if np.sum(mask) == 0:
        return np.array([230, 230, 230], dtype=np.uint8)

    bg_color = np.median(img_bgr[mask], axis=0)
    return bg_color.astype(np.uint8)

In [14]:
def fill_top_bottom_with_background(img_bgr, cut_top=15, cut_bottom=0, bright_thresh=200):
    """
    Fill top and optional bottom strip with estimated background colour.
    """
    out = img_bgr.copy()
    bg_color = estimate_background_color(out, bright_thresh=bright_thresh)

    if cut_top > 0:
        out[:cut_top, :] = bg_color

    if cut_bottom > 0:
        out[-cut_bottom:, :] = bg_color

    return out

In [26]:
def has_enough_ink(img_bgr, ink_thresh=100, min_ink_pixels=50):
    """
    Ink check using grayscale version.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    ink_pixels = np.sum(gray < ink_thresh)
    return ink_pixels > min_ink_pixels


In [33]:
def filter_line_segments_rgb(input_folder, output_folder, page_list, cut_top=1, cut_bottom=0):
    os.makedirs(output_folder, exist_ok=True)

    for page_name in page_list:
        page_base = os.path.splitext(page_name)[0]
        page_folder = os.path.join(input_folder, page_base)

        if not os.path.isdir(page_folder):
            print(f"Skipping {page_name}: folder not found")
            continue

        out_page_folder = os.path.join(output_folder, page_base)
        os.makedirs(out_page_folder, exist_ok=True)

        for line_file in os.listdir(page_folder):
            if not line_file.lower().endswith(".jpg"):
                continue

            line_path = os.path.join(page_folder, line_file)

            # Read original colour image
            img = cv2.imread(line_path, cv2.IMREAD_COLOR)

            if img is None:
                print(f"Could not read {line_path}")
                continue

          

            # 1. fill top  with original background colour
            img = fill_top_bottom_with_background(
                img,
                cut_top=cut_top,
                cut_bottom=cut_bottom,
                bright_thresh=120
            )

            

            # 2. check whether real ink remains
            if has_enough_ink(img, ink_thresh=180, min_ink_pixels=120):
                out_path = os.path.join(out_page_folder, line_file)
                cv2.imwrite(out_path, img)
            else:
                print(f"Excluded: {page_name}/{line_file} (too little ink left)")

In [31]:
def filter_single_line_segment_rgb(input_path, output_path, cut_top=1, cut_bottom=1):
    img = cv2.imread(input_path, cv2.IMREAD_COLOR)

    if img is None:
        print(f"Could not read {input_path}")
        return False


    # 1. fill top and optional bottom with estimated background colour
    img = fill_top_bottom_with_background(
        img,
        cut_top=cut_top,
        cut_bottom=cut_bottom,
        bright_thresh=120
    )

   

    # 2. check whether enough ink remains
    if has_enough_ink(img, ink_thresh=180, min_ink_pixels=120):
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        cv2.imwrite(output_path, img)
        print(f"Saved: {output_path}")
        return True
    else:
        print(f"Excluded: {input_path} (too little ink left)")
        return False

In [32]:
input_path = r'seam_carved_line_images6_case_and_ayer_imgs\995745818805867_case_ms_4a_31_004-1\995745818805867_case_ms_4a_31_004-1_line_001.jpg'
output_path = r"test_single_colored_line_seg\995745818805867_case_ms_4a_31_004-1_line_001.jpg"

filter_single_line_segment_rgb(
    input_path=input_path,
    output_path=output_path,
    cut_top=5,
    cut_bottom=0
)

Saved: test_single_colored_line_seg\995745818805867_case_ms_4a_31_004-1_line_001.jpg


True

In [34]:
input_folder = r"seam_carved_line_images6_case_and_ayer_imgs"
output_folder = r"filtered_case_and_ayer_line_images_over21_rgb"

filter_line_segments_rgb(
    input_folder=input_folder,
    output_folder=output_folder,
    page_list=df1["image"],
    cut_top=5,
    cut_bottom=0
)

Excluded: 995745818805867_case_ms_4a_31_055-1.jpg/995745818805867_case_ms_4a_31_055-1_line_022.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a_31_066.jpg/995745818805867_case_ms_4a_31_066_line_023.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a_31_077-1.jpg/995745818805867_case_ms_4a_31_077-1_line_024.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a_31_089-1.jpg/995745818805867_case_ms_4a_31_089-1_line_020.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a_31_089-1.jpg/995745818805867_case_ms_4a_31_089-1_line_021.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a_31_089-1.jpg/995745818805867_case_ms_4a_31_089-1_line_022.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a_31_089-1.jpg/995745818805867_case_ms_4a_31_089-1_line_023.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a_31_093-0.jpg/995745818805867_case_ms_4a_31_093-0_line_022.jpg (too little ink left)
Excluded: 995745818805867_case_ms_4a

# Good now we do a linearized spliting of text to image line segments.

In [4]:
# normalized transcript in filtered_output.csv

df2 = pd.read_csv(r'C:\Users\pmlma\.vscode\Math_804_transcription_project\csv_files\filtered_output.csv')

df2.head()

,filename,transcription
0,995745818805867_case_ms_4a_31_003-1.tif,"Home. June 6, 1869 Chicago U.S.A. In the libra..."
1,995745818805867_case_ms_4a_31_004-0.tif,2that is a subject I can not talk nor write ab...
2,995745818805867_case_ms_4a_31_004-1.tif,3Florence Arnold raced over to inquire how we ...
3,995745818805867_case_ms_4a_31_005-0.tif,"Tuesday, entirely on my account; it seems he &..."
4,995745818805867_case_ms_4a_31_005-1.tif,"then thirteen) that ""it would be better if Dud..."


In [5]:
#match filename is df1 to filter df2
df1_bases = df1['image'].str.replace(r'\.jpg$', '', regex=True)
df2_bases = df2['filename'].str.replace(r'\.tif$', '', regex=True)

df2_filtered = df2[df2_bases.isin(df1_bases)]
df2_filtered.head()

,filename,transcription
1,995745818805867_case_ms_4a_31_004-0.tif,2that is a subject I can not talk nor write ab...
3,995745818805867_case_ms_4a_31_005-0.tif,"Tuesday, entirely on my account; it seems he &..."
4,995745818805867_case_ms_4a_31_005-1.tif,"then thirteen) that ""it would be better if Dud..."
5,995745818805867_case_ms_4a_31_006-0.tif,"town, and that they must continue to come here..."
6,995745818805867_case_ms_4a_31_006-1.tif,"not, for Nene told me at the door that Johnny ..."


In [6]:
df2_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 161 entries, 1 to 235
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   filename       161 non-null    object
 1   transcription  161 non-null    object
dtypes: object(2)
memory usage: 3.8+ KB


### Converting transcription to tokens

In [7]:
import re

def normalize_transcript(text):
    if pd.isna(text):
        return ""
    text = text.replace("&", " & ")
    text = text.replace("+", " & ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [8]:

df2_filt_norm = df2_filtered[['filename', 'transcription']].copy()
df2_filt_norm['transcription_normalized'] = df2_filt_norm['transcription'].apply(normalize_transcript)
df2_filt_norm['tokens'] = df2_filt_norm['transcription_normalized'].apply(list) # change . split()  to apply.(list) 
df2_filt_norm['base'] = df2_filt_norm['filename'].str.rsplit('.', n=1).str[0]

print(df2_filt_norm.head())

page_to_tokens = dict(zip(df2_filt_norm['base'], df2_filt_norm['tokens']))

                                  filename  \
1  995745818805867_case_ms_4a_31_004-0.tif   
3  995745818805867_case_ms_4a_31_005-0.tif   
4  995745818805867_case_ms_4a_31_005-1.tif   
5  995745818805867_case_ms_4a_31_006-0.tif   
6  995745818805867_case_ms_4a_31_006-1.tif   

                                       transcription  \
1  2that is a subject I can not talk nor write ab...   
3  Tuesday, entirely on my account; it seems he &...   
4  then thirteen) that "it would be better if Dud...   
5  town, and that they must continue to come here...   
6  not, for Nene told me at the door that Johnny ...   

                            transcription_normalized  \
1  2that is a subject I can not talk nor write ab...   
3  Tuesday, entirely on my account; it seems he &...   
4  then thirteen) that "it would be better if Dud...   
5  town, and that they must continue to come here...   
6  not, for Nene told me at the door that Johnny ...   

                                              tok

### Match pages to folders and split page tokens across lines

In [15]:
#Matching page folders to tokens

def get_ordered_line_files(page_folder):
    line_files = []
    for fname in os.listdir(page_folder):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')) and '_line_' in fname:
            line_files.append(fname)

    def extract_line_num(fname):
        m = re.search(r'_line_(\d+)', fname)
        return int(m.group(1)) if m else 999999

    line_files.sort(key=extract_line_num)
    return line_files

def match_pagefolders_to_tokens(input_folder, df_tokens):
    matched = []

    for _, row in df_tokens.iterrows():
        page_base = row['base']
        tokens = row['tokens']

        page_folder = os.path.join(input_folder, page_base)

        if not os.path.isdir(page_folder):
            print(f"Folder not found for page: {page_base}")
            continue

        line_files = get_ordered_line_files(page_folder)

        matched.append({
            'page': page_base,
            'folder': page_folder,
            'line_files': line_files,
            'tokens': tokens
        })

    return matched


In [10]:
def line_ink_score(image_path):
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(gray, 175, 255, cv2.THRESH_BINARY_INV)
    if binary is None:
        return 1.0
    ink = np.sum(binary > 200)
    return max(float(ink), 1.0)

In [11]:
def find_next_boundary(tokens, proposed_end, max_end):
    """
    Move proposed_end forward until the previous character is a boundary.
    Boundary means the line ends after a space or hyphen.
    """
    end = proposed_end

    while end < max_end:
        if tokens[end - 1] in [" ", "-"]:
            return end
        end += 1

    return max_end

In [12]:
def split_char_tokens_across_lines(page_folder, line_files, tokens):
    if not line_files:
        return []

    if not tokens:
        return [
            {
                'line_file': lf,
                'assigned_tokens': [],
                'assigned_text': ''
            }
            for lf in line_files
        ]

    scores = [line_ink_score(os.path.join(page_folder, lf)) for lf in line_files]
    total_score = sum(scores)
    n_tokens = len(tokens)

    raw_counts = [s / total_score * n_tokens for s in scores]
    counts = [int(round(x)) for x in raw_counts]

    diff = n_tokens - sum(counts)

    if diff != 0:
        fracs = [x - int(x) for x in raw_counts]
        order = np.argsort(fracs)[::-1] if diff > 0 else np.argsort(fracs)

        for i in order[:abs(diff)]:
            counts[i] += 1 if diff > 0 else -1
            if counts[i] < 0:
                counts[i] = 0

    while sum(counts) < n_tokens:
        counts[np.argmax(scores)] += 1

    while sum(counts) > n_tokens:
        idx = np.argmax(counts)
        if counts[idx] > 0:
            counts[idx] -= 1

    results = []
    start = 0

    for idx, (lf, c) in enumerate(zip(line_files, counts)):
        proposed_end = start + c

        # last line gets everything left
        if idx == len(line_files) - 1:
            end = n_tokens
        else:
            proposed_end = min(proposed_end, n_tokens)

            # keep assigning until previous char is space or hyphen
            end = find_next_boundary(tokens, proposed_end, n_tokens)

        assigned = tokens[start:end]

        results.append({
            'line_file': lf,
            'assigned_tokens': assigned,
            'assigned_text': ''.join(assigned)
        })

        start = end

    return results

In [16]:
# 1. match pages to folders
matched_pages = match_pagefolders_to_tokens(
    input_folder="filtered_case_and_ayer_line_images_over21_rgb", 
    df_tokens=df2_filt_norm[['base', 'tokens']]
)

# 2. split page tokens across lines
all_line_assignments = []

for item in matched_pages:
    split_result = split_char_tokens_across_lines(
        item['folder'],
        item['line_files'],
        item['tokens']
    )

    for row in split_result:
        all_line_assignments.append({
            'page': item['page'],
            'line_file': row['line_file'],
            'assigned_text': row['assigned_text'],
            'assigned_tokens': row['assigned_tokens']
        })

df_line_assignments = pd.DataFrame(all_line_assignments)
print(df_line_assignments.head())

# 3. save
df_line_assignments.to_csv("line_level_linearized_labels3.csv", index=False)

                                  page  \
0  995745818805867_case_ms_4a_31_004-0   
1  995745818805867_case_ms_4a_31_004-0   
2  995745818805867_case_ms_4a_31_004-0   
3  995745818805867_case_ms_4a_31_004-0   
4  995745818805867_case_ms_4a_31_004-0   

                                          line_file  \
0  995745818805867_case_ms_4a_31_004-0_line_001.jpg   
1  995745818805867_case_ms_4a_31_004-0_line_002.jpg   
2  995745818805867_case_ms_4a_31_004-0_line_003.jpg   
3  995745818805867_case_ms_4a_31_004-0_line_004.jpg   
4  995745818805867_case_ms_4a_31_004-0_line_005.jpg   

                              assigned_text  \
0    2that is a subject I can not talk nor    
1     write about either. Our breakfast of    
2     pigeons, fried potatoes & corn-bread    
3       tasted deliciously, as did also my    
4  favorite dinner; roast-beef, asparagus,    

                                     assigned_tokens  
0  [2, t, h, a, t,  , i, s,  , a,  , s, u, b, j, ...  
1  [w, r, i, t, e,  , a

In [17]:
df_line_assignments.head()

,page,line_file,assigned_text,assigned_tokens
0,995745818805867_case_ms_4a_31_004-0,995745818805867_case_ms_4a_31_004-0_line_001.jpg,2that is a subject I can not talk nor,"[2, t, h, a, t, , i, s, , a, , s, u, b, j, ..."
1,995745818805867_case_ms_4a_31_004-0,995745818805867_case_ms_4a_31_004-0_line_002.jpg,write about either. Our breakfast of,"[w, r, i, t, e, , a, b, o, u, t, , e, i, t, ..."
2,995745818805867_case_ms_4a_31_004-0,995745818805867_case_ms_4a_31_004-0_line_003.jpg,"pigeons, fried potatoes & corn-bread","[p, i, g, e, o, n, s, ,, , f, r, i, e, d, , ..."
3,995745818805867_case_ms_4a_31_004-0,995745818805867_case_ms_4a_31_004-0_line_004.jpg,"tasted deliciously, as did also my","[t, a, s, t, e, d, , d, e, l, i, c, i, o, u, ..."
4,995745818805867_case_ms_4a_31_004-0,995745818805867_case_ms_4a_31_004-0_line_005.jpg,"favorite dinner; roast-beef, asparagus,","[f, a, v, o, r, i, t, e, , d, i, n, n, e, r, ..."


### Copying all line segment images listed in df into one flat folder.

In [18]:

import shutil

def extract_all_lines_to_one_folder(df, source_root, output_folder):
    """
    Copies all line segment images listed in df into one flat folder.

    Expected dataframe columns:
        - page
        - line_file

    Source structure:
        source_root/page/line_file

    Output filename:
        page__line_file
    """

    os.makedirs(output_folder, exist_ok=True)

    copied = 0
    missing = 0

    for _, row in df.iterrows():
        page = row["page"]
        line_file = row["line_file"]

        src_path = os.path.join(source_root, page, line_file)

        if not os.path.exists(src_path):
            print(f"Missing: {src_path}")
            missing += 1
            continue

        new_name = f"{line_file}"
        dst_path = os.path.join(output_folder, new_name)

        shutil.copy2(src_path, dst_path)
        copied += 1

    print("=" * 50)
    print(f"Copied files : {copied}")
    print(f"Missing files: {missing}")
    print(f"Output folder: {output_folder}")

In [19]:
source_root = r"filtered_case_and_ayer_line_images_over21_rgb"
output_folder = r"all_case_and_ayer_line_segments_folder"

extract_all_lines_to_one_folder(
    df=df_line_assignments,
    source_root=source_root,
    output_folder=output_folder
)



Copied files : 3701
Missing files: 0
Output folder: all_case_and_ayer_line_segments_folder


def binarize_for_cc(gray):
    clahe = cv2.createCLAHE(clipLimit=3.3, tileGridSize=(8, 8))
    enhance = clahe.apply(gray)
    blurred = cv2.GaussianBlur(enhance, (3, 3), 0)

    binary = cv2.adaptiveThreshold(
        blurred,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        12
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 1))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    return binary

# Side by side plot of 1st 5 line segments


In [ ]:
import pandas as pd
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
img1 = r'C:\Users\pmlma\Downloads\1st_5_segments_case_ms_4a_31_004-0_seams.jpg'
img2 = r'C:\Users\pmlma\Downloads\Screenshot 2026-05-09 134225.png'